# XGrammar Hands-on Lab
## Programmatic Usage & Engine Integrations
**Companion notebook to the 90-minute lecture *Inside XGrammar*.**

> Bilingual notes: each section has a short Spanish summary in *italics*.

This notebook walks through:

1. The **public Python API** — `Grammar`, `TokenizerInfo`, `GrammarCompiler`, `CompiledGrammar`, `GrammarMatcher`.
2. The **easy path** with HuggingFace `transformers` (one-line integration).
3. The **transparent path** — the manual matcher loop that maps 1:1 to the theory blocks.
4. Integrations with **vLLM**, **SGLang**, and **MLC-LLM** (reference snippets; these need a GPU box or appropriate hardware).
5. A **decision matrix** for picking the right engine.

*Cuaderno bilingüe: cada sección incluye un breve resumen en español. Las secciones 1–3 corren en CPU con un modelo pequeño; las 4–6 son snippets de referencia para correr en una máquina con GPU o hardware adecuado.*


## 1. Setup · *Instalación*

Install XGrammar + Transformers + Torch. The `xgrammar` wheel is fast (~30s) because all the heavy lifting is in pre-built C++.

*Instala XGrammar, Transformers y PyTorch. El instalador es rápido porque el código C++ ya viene compilado.*


In [1]:
# Run once. Comment out if you've already installed.
%pip install --quiet "xgrammar>=0.1.30" "transformers>=4.45" "torch>=2.2" "pydantic>=2"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 MB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 81.3 MB/s eta 0:00:00


In [2]:
import xgrammar as xgr
import torch
# print("xgrammar:", xgr.__version__) # Removed as xgrammar module has no __version__ attribute
print("torch:   ", torch.__version__)
print("CUDA:    ", torch.cuda.is_available())

torch:    2.10.0+cu128
CUDA:     True


## 2. The public API in five classes · *La API pública en cinco clases*

The mental model from the lecture maps 1:1 to the Python API:

| Class | What it represents | When it runs |
|---|---|---|
| `Grammar` | What you want — JSON, JSON schema, regex, or EBNF | once |
| `TokenizerInfo` | Vocabulary + chat template + special tokens | once per tokenizer |
| `GrammarCompiler` | Builds the PDA and the adaptive token mask cache | once per (grammar, tokenizer) |
| `CompiledGrammar` | Pickleable, reusable handle to the cache | shared across requests |
| `GrammarMatcher` | The runtime — `accept_token`, `fill_next_token_bitmask` | **every decoding step** |

*Las primeras cuatro se ejecutan una sola vez. Solo `GrammarMatcher` corre por paso de decodificación.*


## 3. The easy path — HuggingFace Transformers · *El camino fácil*

This is the one-line integration. We load Llama-3.2-1B-Instruct (small enough to run on CPU), compile the built-in JSON grammar, and pass an `xgr.contrib.hf.LogitsProcessor` to `model.generate()`.

*La integración mínima: cargamos un modelo pequeño, compilamos la gramática y pasamos un `LogitsProcessor` a `model.generate()`. La máscara se aplica invisiblemente.*

> Heads up: the first `model.generate` call on CPU with the 1B model takes ~20–40s. The structural constraint is enforced for free.


In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig

# The original model 'meta-llama/Llama-3.2-1B-Instruct' is a gated repo and requires explicit access from Hugging Face.
# As a workaround, we'll use a publicly available model. You can switch back if you gain access.
MODEL = "gpt2-medium"
# If you have access to gated models and have set your Hugging Face token, you might use:
# MODEL = "meta-llama/Llama-3.2-1B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float32, device_map=device)
tokenizer = AutoTokenizer.from_pretrained(MODEL)
config = AutoConfig.from_pretrained(MODEL)
print(f"Loaded {MODEL} on {device}. Vocab size: {config.vocab_size}")

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-medium
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded gpt2-medium on cuda. Vocab size: 50257


In [4]:
# Compile the built-in JSON grammar. This is the only XGrammar work that happens once.
tokenizer_info = xgr.TokenizerInfo.from_huggingface(tokenizer, vocab_size=config.vocab_size)
compiler = xgr.GrammarCompiler(tokenizer_info)
compiled = compiler.compile_builtin_json_grammar()
print("Grammar compiled. CompiledGrammar object:", compiled)


Grammar compiled. CompiledGrammar object: <xgrammar.compiler.CompiledGrammar object at 0x7c100635bec0>


In [7]:
# Prompt the model and decode with the grammar enforced.
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Introduce yourself in JSON briefly. Use keys: name, role, fun_fact."},
]

# The gpt2-medium tokenizer does not have a chat_template. Construct the prompt manually.
# For chat models, you would use tokenizer.apply_chat_template.
# For gpt2, we'll concatenate the messages. A simple approach for demonstration:
prompt_text = "System: " + messages[0]["content"] + "\nUser: " + messages[1]["content"] + "\nAssistant:"

inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

logits_processor = xgr.contrib.hf.LogitsProcessor(compiled)
out_ids = model.generate(
    **inputs,
    max_new_tokens=200,
    logits_processor=[logits_processor],
    do_sample=False,
)
generated = tokenizer.decode(out_ids[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print(generated)

# Sanity check: the output parses as JSON.
import json
parsed = json.loads(generated)
print("\n✓ Parsed JSON keys:", list(parsed.keys()))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


{ "name": "John", "role": "Assistant", "fun_fact": "I'm a nice guy" }

✓ Parsed JSON keys: ['name', 'role', 'fun_fact']


### 3a. Constraining with a JSON Schema · *Con esquema JSON*

`compile_builtin_json_grammar` accepts any JSON. To pin a specific shape, give the compiler a JSON Schema string.

*Para forzar una forma específica, dale al compilador un string con un JSON Schema.*


In [9]:
schema_str = """
{
  "type": "object",
  "properties": {
    "name": {"type": "string"},
    "age": {"type": "integer", "minimum": 0, "maximum": 200},
    "skills": {"type": "array", "items": {"type": "string"}, "minItems": 1}
  },
  "required": ["name", "age", "skills"]
}
"""
compiled_schema = compiler.compile_json_schema(schema_str)
logits_processor = xgr.contrib.hf.LogitsProcessor(compiled_schema)

messages = [
    {"role": "user", "content": "Describe Ada Lovelace as JSON with name, age (at death), and skills."},
]

# The gpt2-medium tokenizer does not have a chat_template. Construct the prompt manually.
# For chat models, you would use tokenizer.apply_chat_template.
# For gpt2, we'll concatenate the messages. A simple approach for demonstration:
prompt_text = "User: " + messages[0]["content"] + "\nAssistant:"

inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

out_ids = model.generate(**inputs, max_new_tokens=200, logits_processor=[logits_processor], do_sample=False)
print(tokenizer.decode(out_ids[0][inputs.input_ids.shape[1]:], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


{ "name": "Ada Lovelace", "age": 30, "skills": [ "computer programming", "computer science", "computer engineering", "computer science", "computer engineering", "computer science", "computer engineering", "computer science", "computer engineering", "computer science", "computer engineering", "computer science", "computer engineering", "computer science", "computer engineering", "computer science", "computer engineering", "computer science", "computer engineering", "computer science", "computer engineering", "computer science", "computer engineering", "computer science", "computer engineering", "computer science", "computer engineering", "computer science", "computer engineering", "computer science", "computer engineering", "computer science", "computer engineering", "computer science", "computer engineering", "computer science", "computer engineering", "computer science", "computer engineering", "computer science", "computer engineering", "computer science", "computer engineering", "co

### 3b. Constraining with a custom EBNF · *Con EBNF personalizado*

Define a tiny domain-specific language (DSL) and force the model into it. This is where XGrammar shines vs. plain JSON-mode APIs.

*Define una mini-DSL con EBNF y obliga al modelo a respetarla. Aquí es donde XGrammar brilla sobre las APIs de solo-JSON.*


In [12]:
ebnf = r"""
root      ::= sentiment
sentiment ::= "positive" | "negative" | "neutral"
"""
compiled_ebnf = compiler.compile_grammar(ebnf)
logits_processor = xgr.contrib.hf.LogitsProcessor(compiled_ebnf)

messages = [{"role": "user", "content": "Classify the sentiment of: 'I love this course.' Output one word."}]

# The gpt2-medium tokenizer does not have a chat_template. Construct the prompt manually.
prompt_text = "User: " + messages[0]["content"] + "\nAssistant:"

inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

out_ids = model.generate(**inputs, max_new_tokens=20, logits_processor=[logits_processor], do_sample=False)
print(tokenizer.decode(out_ids[0][inputs.input_ids.shape[1]:], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


neutral


## 4. The transparent path — manual matcher loop · *Loop manual*

This is the same algorithm `LogitsProcessor` wraps. Showing it explicitly so you can see the theory in action:

- `allocate_token_bitmask(1, vocab)` → the `(batch, ⌈|V|/32⌉)` int32 bitmask from Block B.
- `fill_next_token_bitmask` → adaptive cache lookup + check on the ~120 context-dependent tokens (Block E).
- `apply_token_bitmask_inplace` → set rejected token logits to `-inf` before softmax (Block B).
- `accept_token` → advance the persistent execution stack (Block F).

*Este es el algoritmo que `LogitsProcessor` envuelve. Lo mostramos explícito para ver la teoría en acción.*


In [13]:
matcher = xgr.GrammarMatcher(compiled_schema)
token_bitmask = xgr.allocate_token_bitmask(1, tokenizer_info.vocab_size)
print(f"Bitmask shape: {tuple(token_bitmask.shape)}, dtype: {token_bitmask.dtype}")
print(f"Bytes per request per step: {token_bitmask.element_size() * token_bitmask.numel()} B")


Bitmask shape: (1, 1571), dtype: torch.int32
Bytes per request per step: 6284 B


In [15]:
# A bare-bones decode loop. Inefficient (re-feeds full sequence) but illustrative.
messages = [{"role": "user", "content": "Describe Ada Lovelace as JSON with name, age, and skills."}]

# The gpt2-medium tokenizer does not have a chat_template. Construct the prompt manually.
prompt_text = "User: " + messages[0]["content"] + "\nAssistant:"

inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
input_ids = inputs.input_ids[0].tolist()
prompt_len = len(input_ids)

matcher.reset()
step = 0
max_steps = 200

while not matcher.is_terminated() and step < max_steps:
    logits = model(torch.tensor([input_ids], device=model.device)).logits
    matcher.fill_next_token_bitmask(token_bitmask)
    xgr.apply_token_bitmask_inplace(logits[0, -1, :], token_bitmask.to(logits.device))
    next_id = torch.argmax(torch.softmax(logits[0, -1, :], dim=-1)).item()
    matcher.accept_token(next_id)
    input_ids.append(next_id)
    step += 1

print(f"Generated {step} tokens.")
print(tokenizer.decode(input_ids[prompt_len:], skip_special_tokens=True))

Generated 200 tokens.
{ "name": "Ada Lovelace", "age": 30, "skills": [ "computer programming", "computer science", "computer engineering", "computer science", "computer engineering", "computer science", "computer engineering", "computer science", "computer engineering", "computer science", "computer engineering", "computer science", "computer engineering", "computer science", "computer engineering", "computer science", "computer engineering", "computer science", "computer engineering", "computer science", "computer engineering", "computer science", "computer engineering", "computer science", "computer engineering", "computer science", "computer engineering", "computer science", "computer engineering", "computer science", "computer engineering", "computer science", "computer engineering", "computer science", "computer engineering", "computer science", "computer engineering", "computer science", "computer engineering", "computer science", "computer engineering", "computer science", "comp

**Why call out the manual loop in class?** It makes the abstraction tangible:

- The bitmask is just a small `int32` tensor — students can `print` it.
- `accept_token` is a single function call but it's the rollback-friendly persistent stack underneath.
- Every concept from Block E and Block F has a one-line Python equivalent.

*¿Por qué mostrar el loop manual? Hace tangible la abstracción: cada concepto teórico tiene una línea de Python equivalente.*


## 5. Production serving with vLLM · *Serving con vLLM*

vLLM is the most common production deployment for XGrammar (it's the default `guided_decoding_backend`). Two interfaces: **offline** (in-process `LLM`) and **OpenAI-compatible server** (drop-in for any OpenAI SDK).

*vLLM es el despliegue más común en producción para XGrammar. Tiene dos modos: offline (en proceso) y servidor compatible con OpenAI.*

> The cells below assume a CUDA GPU and `pip install vllm`. If you don't have one, read the snippets; the structure is identical.


In [2]:
# Run on a GPU box.
!pip install vllm openai

# --- Offline ---
from vllm import LLM, SamplingParams
from vllm.sampling_params import GuidedDecodingParams
#
llm = LLM(model="meta-llama/Llama-3.2-1B-Instruct")
guided = GuidedDecodingParams(json=schema_str, backend="xgrammar")
sp = SamplingParams(temperature=0.7, max_tokens=128, guided_decoding=guided)
out = llm.generate(["Describe Ada Lovelace as JSON."], sp)
print(out[0].outputs[0].text)
print("vLLM offline pattern — uncomment and run on a GPU box.")

ModuleNotFoundError: No module named 'vllm.guided_decoding'

In [ ]:
# --- OpenAI-compatible server ---
# Start in a shell:
#   vllm serve meta-llama/Llama-3.2-1B-Instruct --guided-decoding-backend xgrammar
#
# Then in Python:
# from openai import OpenAI
# client = OpenAI(base_url="http://localhost:8000/v1", api_key="-")
# resp = client.chat.completions.create(
#     model="meta-llama/Llama-3.2-1B-Instruct",
#     messages=[{"role": "user", "content": "Describe Ada Lovelace as JSON."}],
#     extra_body={"guided_json": schema_str, "guided_decoding_backend": "xgrammar"},
# )
# print(resp.choices[0].message.content)
print("vLLM server pattern — uncomment after `vllm serve` is up.")


Useful `extra_body` keys for the OpenAI-compatible server:

- `guided_json` — JSON schema
- `guided_regex` — regular expression
- `guided_choice` — finite set of strings (`["yes", "no"]`)
- `guided_grammar` — full EBNF (use this for SQL, function-call DSLs, etc.)
- `guided_decoding_backend` — force `"xgrammar"` (default) or fall back to `"outlines"` / `"lm-format-enforcer"`


## 6. SGLang

SGLang uses the same XGrammar backend but exposes the constraints as sampling parameters (`json_schema`, `regex`, `ebnf`, `structural_tag`). It also has the best support for **agentic patterns** where the model emits free text interleaved with tool calls.

*SGLang usa el mismo backend pero expone las restricciones como parámetros de muestreo. Tiene el mejor soporte para patrones agénticos con llamadas a tools.*


In [ ]:
# Run on a GPU box.
# !pip install sglang

# import sglang as sgl, json
# from pydantic import BaseModel
#
# class Capital(BaseModel):
#     name: str
#     population: int
#
# llm = sgl.Engine(
#     model_path="meta-llama/Meta-Llama-3.1-8B-Instruct",
#     grammar_backend="xgrammar",   # default
# )
# sampling_params = {
#     "temperature": 0.1,
#     "top_p": 0.95,
#     "json_schema": json.dumps(Capital.model_json_schema()),
#     # Try also:
#     # "regex": "(France|England)",
#     # "ebnf":  "root ::= 'yes' | 'no'",
# }
# outputs = llm.generate(["Capital of France, JSON please."], sampling_params)
# print(outputs[0]["text"])
# llm.shutdown()
print("SGLang pattern — uncomment and run on a GPU box.")


## 7. MLC-LLM — Apple Silicon & browser · *MLC-LLM: Apple Silicon y navegador*

MLC-LLM is from the same team as XGrammar. The Python `MLCEngine` mirrors the OpenAI Chat Completions API. The same engine compiles down to WebGPU via WebLLM — the only path in this list that lets you ship constrained decoding in a browser tab.

*MLC-LLM es del mismo equipo que XGrammar. El `MLCEngine` en Python imita la API de OpenAI. El mismo motor compila a WebGPU vía WebLLM — la única vía aquí para llevar decodificación restringida al navegador.*


In [ ]:
# Run on Apple Silicon or a CUDA box.
# !pip install --pre -U -f https://mlc.ai/wheels mlc-llm-nightly mlc-ai-nightly

# from mlc_llm import MLCEngine
#
# engine = MLCEngine("HF://mlc-ai/Llama-3.2-1B-Instruct-q4f16_1-MLC")
# resp = engine.chat.completions.create(
#     messages=[{"role": "user", "content": "Describe Ada Lovelace as JSON."}],
#     response_format={"type": "json_object", "schema": schema_str},
# )
# print(resp.choices[0].message.content)
# engine.terminate()
print("MLC-LLM pattern — uncomment after installing the MLC wheels.")


## 8. Which engine when? · *¿Qué motor cuándo?*

| If you want… | Reach for | *En español* |
|---|---|---|
| Minimum-friction educational integration | HF Transformers + `xgr.contrib.hf.LogitsProcessor` | Integración educativa mínima |
| Production serving with the OpenAI API | **vLLM** (default backend) | Serving en producción con OpenAI API |
| Offline batch + structural tags / agents | **SGLang** | Batch offline + agentes |
| Apple Silicon / browser / mobile | **MLC-LLM** | Apple Silicon, navegador, móvil |
| To understand the internals | manual `GrammarMatcher` loop | Para entender las tripas |

**Unifying point**: all four use XGrammar under the hood. The only differences are how the user expresses the constraint (Python kwargs vs. OpenAI `extra_body` vs. `response_format`) and where the engine puts the bitmask in its decode loop. The algorithm is the one you spent 90 minutes learning.

*Punto unificador: los cuatro usan XGrammar por debajo. Solo cambia la superficie — el algoritmo es el mismo.*


## 9. Take-home mini-assignment · *Mini-tarea*

Pick a grammar (JSON schema, regex, or EBNF — your choice). Run it through **two** of the four integrations (HF + one of vLLM / SGLang / MLC-LLM). Diff the outputs and report:

1. Did both engines respect the grammar?
2. What was the time-to-first-token in each?
3. Were there cases where one engine succeeded and the other failed (e.g., on edge characters)?

Submit a one-paragraph write-up the following week.

*Elige una gramática y córrela en dos de las cuatro integraciones. Reporta correctud, latencia y diferencias.*

---

### References

- [XGrammar quick start](https://xgrammar.mlc.ai/docs/start/quick_start) — the HF integration above.
- [vLLM structured outputs](https://docs.vllm.ai/en/latest/features/structured_outputs.html) — `guided_json` / `guided_grammar`.
- [SGLang structured outputs](https://docs.sglang.ai/advanced_features/structured_outputs.html) — `json_schema` / `regex` / `ebnf` / `structural_tag`.
- [MLC-LLM Engine API](https://llm.mlc.ai/docs/) — `response_format`.
- [Workflow of XGrammar](https://xgrammar.mlc.ai/docs/tutorials/workflow_of_xgrammar.html) — the full pipeline you saw in the lecture.
